# Application to the two clinical datasetsLoads the CSVs written by the R preprocessing scripts, fits the three AFTmodels, and merges the Cox linear predictors exported by the R scripts.## What changed in this notebookThe model, loss, sampler, metrics and data generation used to be defined**inline in this notebook**, and the same block was copy-pasted into five othernotebooks. That is why two defects survived so long: fixing one copy left theothers untouched, and the copies had already drifted apart.All of that now lives in the `rnn_agt` package. This notebook only sets up anexperiment and reports it.Two fixes are inherited automatically:1. **Censoring now reaches the outcome.** The old `prepare_subjects_for_nn`   passed `subj['log_gaps']` — the *latent, uncensored* gap times — to the   model, while `delta` said some records were censored. Padding past the   censoring point was passed through as real data too.2. **The WRS normalization `1/(K_i* K_l*)` is applied.** The old loss was a   plain Gehan rank loss. The subject-level weight is the mechanism that   handles induced dependent censoring, so without it the estimating function   is biased toward subjects with many events.`simulation/defect_impact.ipynb` measures how much both defects changed the numbers.**Third defect, specific to the application.** The R scripts reported`summary(fit)$concordance` — Harrell's C computed *in sample, unweighted, onthe full dataset*. RNN-AGT's number is out-of-sample and IPCW-weighted. Table 7therefore compared two different estimands. The Cox scripts now export held-outlinear predictors and this notebook scores them with the same estimator usedfor the neural models.

In [ ]:
import os, syssys.path.insert(0, os.path.abspath(".."))   # repository root, so `rnn_agt` importsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport rnn_agtfrom rnn_agt import data as Dfrom rnn_agt.seeds import make_seedsfrom rnn_agt.train import TrainConfig, train_model, predictfrom rnn_agt.metrics import evaluateprint("rnn_agt", rnn_agt.__version__)

In [ ]:
sys.path.insert(0, os.path.abspath("../experiments"))from run_ablation_and_splits import prepare_real_datafrom rnn_agt.cox_bridge import (write_splits, load_cox_predictions,                                score_cox_predictions, merge_into_outcomes,                                check_orientation, COX_MODELS, COX_LABELS)from rnn_agt.evaluation import run_repeated_splits, summarise, paired_differenceDATA = {    "cgd": "../data/cgd.csv",    "crc": "../data/crc.csv",}datasets = {}for name, path in DATA.items():    subs, p, ids = prepare_real_data(path, return_ids=True)    datasets[name] = (subs, p, ids)    n_rec = sum(len(s["log_gaps"]) for s in subs)    n_ev = sum(int(s["delta"].sum()) for s in subs)    print(f"{name}: {len(subs)} subjects, {n_rec} records, {n_ev} events, p={p}")    print(f"       censoring fraction {1 - n_ev / n_rec:.1%}")

Check these counts against Section 5.1 before going further. The manuscript states n=128 (CGD) and n=403 (CRC).

In [ ]:
N_SPLITS = 200      # reduce while developingSEED = 20260903configs = {    "aft_wrs": TrainConfig(model="aft_wrs", lr=1e-2, epochs=10, pair_sample_s=30),    "nn_aft":  TrainConfig(model="nn_aft",  lr=3e-4, epochs=10, pair_sample_s=30,                           hidden_dim=64, gru_layers=2),    "rnn_agt": TrainConfig(model="rnn_agt", lr=3e-4, epochs=10, pair_sample_s=30,                           hidden_dim=64, gru_layers=2),}os.makedirs("../results/splits", exist_ok=True)splits_dfs = {}for ds, (subs, p, ids) in datasets.items():    path = f"../results/splits/splits_{ds}.csv"    splits_dfs[ds] = write_splits(subs, ids, path, SEED,                                  n_splits=N_SPLITS, test_frac=0.30)    print(f"{ds}: split assignments -> {path}")

### Run the R Cox scripts nowThe split files are written. From the repository root:```bashRscript "application/Dataset2_Classical_Recurrent_Event_Models.R" \    --data data/cgd.csv \    --splits results/splits/splits_cgd.csv \    --out results/cox_lp_cgd.csvRscript "application/Dataset1_Classical_Recurrent_Event_Models.R" \    --data data/crc.csv \    --splits results/splits/splits_crc.csv \    --out results/cox_lp_crc.csv```The cells below run with or without those files; the Cox rows are simply blankif they are absent.

In [ ]:
all_outcomes = {}for ds, (subs, p, ids) in datasets.items():    print(f"\n=== {ds} ===")    outcomes = run_repeated_splits(        subs, p, configs, SEED, n_splits=N_SPLITS, test_frac=0.30,        progress=lambda d, t: print(f"  split {d}/{t}", flush=True) if d % 20 == 0 else None,    )    cox_path = f"../results/cox_lp_{ds}.csv"    if os.path.exists(cox_path):        scores = score_cox_predictions(load_cox_predictions(cox_path),                                       subs, ids, splits_dfs[ds])        warn = check_orientation(scores)        if warn:            print(f"  WARNING: {warn}")        outcomes = merge_into_outcomes(outcomes, scores)        print(f"  merged Cox scores from {cox_path}")    else:        print(f"  no Cox predictors at {cox_path}; those rows stay blank")    all_outcomes[ds] = outcomes

In [ ]:
rows = []for ds, outcomes in all_outcomes.items():    summ = summarise(outcomes, "test_cindex")    for model, st in summ.items():        rows.append({"dataset": ds, "model": COX_LABELS.get(model, model),                     "mean C": st["mean"], "sd": st["sd"],                     "2.5%": st["lo"], "97.5%": st["hi"]})pd.DataFrame(rows).round(3).sort_values(["dataset", "mean C"], ascending=[True, False])

### Paired incrementsThe quantity that matters is the paired difference and its spread across splits, not the ranking of marginal means. Read the win rate alongside the mean: a positive mean with a win rate near 0.5 means the capability is not reliably helping.

In [ ]:
for ds, outcomes in all_outcomes.items():    print(f"\n{ds}")    for label, a, b in (        ("delta nonlinearity", "nn_aft", "aft_wrs"),        ("delta history",      "rnn_agt", "nn_aft"),    ):        d = paired_difference(outcomes, a, b, "test_cindex")        print(f"  {label:20s} {d['mean']:+.3f} "              f"(95% CI {d['lo']:+.3f}, {d['hi']:+.3f})  win rate {d['win_rate']:.2f}")    available = [m for m in COX_MODELS if m in outcomes[0].metrics]    if available:        best = max(available,                   key=lambda m: summarise(outcomes, "test_cindex")[m]["mean"])        d = paired_difference(outcomes, "rnn_agt", best, "test_cindex")        print(f"  vs {COX_LABELS[best]:17s} {d['mean']:+.3f} "              f"(95% CI {d['lo']:+.3f}, {d['hi']:+.3f})  win rate {d['win_rate']:.2f}")